# 4D-STEM pyxem/HyperSpy Workflow

Notebook-first workflow for large 4D-STEM data. It runs on a synthetic dataset by default, then can be pointed at real `.mib`, `.npy`, or `.npz` data in `configs/default_workflow.yaml`.

Main stages:
1. environment and configuration
2. MIB-first lazy loading or synthetic fallback
3. dataset inspection
4. virtual BF/ADF/HAADF and COM images
5. radial fingerprints and phase screening
6. candidate phase score maps
7. orientation preview with ROI refinement
8. export summary

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").exists():
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
else:
    PROJECT_ROOT = PROJECT_ROOT.parent
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np

from fourdstem_pipeline import (
    build_annular_masks,
    compute_radial_fingerprints,
    compute_virtual_images,
    load_dataset,
    load_workflow_config,
    run_orientation_preview,
    screen_phases,
)
from fourdstem_pipeline.export import save_summary
from fourdstem_pipeline.plotting import save_image_map, save_profile_plot

print("Python", sys.version)
print("Project root", PROJECT_ROOT)

## Load Configuration

The default config uses `synthetic://demo`. Change `data.path` to a real `.mib` file when the large-data environment is ready.

In [ ]:
config = load_workflow_config(PROJECT_ROOT / "configs" / "default_workflow.yaml")
output_dir = PROJECT_ROOT / config["project"]["output_dir"]
output_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps(config, indent=2))

## Load Dataset

For `.mib`, `load_dataset()` tries HyperSpy lazy loading. For this notebook smoke path, synthetic data keeps the full workflow runnable without real microscope data.

In [ ]:
dataset = load_dataset(
    config["data"]["path"],
    lazy=config["data"].get("lazy", True),
    cache=config["data"].get("cache"),
    chunks=config["data"].get("chunks"),
)
dataset.describe()

## Virtual Images

Virtual detector maps are small 2D outputs, so they are safe first-pass products for large 4D-STEM datasets.

In [ ]:
masks = build_annular_masks(
    dataset.signal_shape,
    config["virtual_images"]["masks"],
    center=config["geometry"].get("center"),
)
virtual = compute_virtual_images(dataset, masks, output_dir=output_dir / "virtual_images")

for name, image in virtual.images.items():
    save_image_map(image, output_dir / "figures" / f"virtual_{name}.png", title=f"Virtual {name.upper()}")
save_image_map(virtual.com_x, output_dir / "figures" / "com_x.png", title="COM X")
save_image_map(virtual.com_y, output_dir / "figures" / "com_y.png", title="COM Y")
save_image_map(virtual.mean_diffraction, output_dir / "figures" / "mean_diffraction.png", title="Mean diffraction", cmap="magma")

{name: image.shape for name, image in virtual.images.items()}

## Radial Fingerprints

Each diffraction pattern is compressed to a compact radial profile. This is the default input for fast phase screening.

In [ ]:
fingerprints = compute_radial_fingerprints(
    dataset,
    config["geometry"],
    config["geometry"]["radial_bins"],
    output_dir=output_dir / "fingerprints",
)
print("fingerprint shape", fingerprints.profiles.shape)
print("radial bins", fingerprints.radii.shape)

## Phase Screening

This stage combines PCA/NMF-style compression, clustering, and optional candidate phase profile scoring. Candidate CIF/template simulation can later populate `reference_profile` fields in the config.

In [ ]:
phase_cfg = config["phase_screening"]

# Demo candidate profiles are built from opposite corners when no external references are configured.
candidate_phases = phase_cfg.get("candidate_phases", [])
if candidate_phases and all(p.get("reference_profile") is None for p in candidate_phases):
    candidate_phases[0]["reference_profile"] = fingerprints.profiles[:4, :4].mean(axis=(0, 1)).tolist()
    if len(candidate_phases) > 1:
        candidate_phases[1]["reference_profile"] = fingerprints.profiles[-4:, -4:].mean(axis=(0, 1)).tolist()

phase = screen_phases(
    fingerprints,
    method=phase_cfg.get("method", "pca_nmf_cluster"),
    candidate_phases=candidate_phases,
    n_components=phase_cfg.get("n_components", 3),
    n_clusters=phase_cfg.get("n_clusters", 3),
    output_dir=output_dir / "phase_screening",
)

save_image_map(phase.labels, output_dir / "figures" / "phase_clusters.png", title="Phase clusters", cmap="tab10")
save_image_map(phase.low_confidence_mask, output_dir / "figures" / "phase_low_confidence.png", title="Phase low confidence", cmap="gray")
save_profile_plot(phase.representative_profiles, output_dir / "figures" / "phase_representative_profiles.png", title="Representative radial profiles")
for name, score in phase.candidate_scores.items():
    save_image_map(score, output_dir / "figures" / f"candidate_score_{name}.png", title=f"Candidate score: {name}")

phase.labels.shape, list(phase.candidate_scores)

## Orientation Preview and ROI Refinement

The first version defaults to a fast full-field preview and ROI refinement. If candidate templates are supplied, this function uses normalized template matching; otherwise it provides a COM-angle proxy map for ROI selection.

In [ ]:
ori_cfg = config["orientation"]
orientation = run_orientation_preview(
    dataset,
    phase_candidates=candidate_phases,
    binning=ori_cfg.get("preview_binning", [2, 2]),
    roi=ori_cfg.get("roi"),
    confidence_threshold=ori_cfg.get("confidence_threshold", 0.05),
    output_dir=output_dir / "orientation",
)

save_image_map(orientation.orientation_index, output_dir / "figures" / "orientation_index.png", title="Orientation index", cmap="twilight")
save_image_map(orientation.score, output_dir / "figures" / "orientation_score.png", title="Orientation score")
save_image_map(orientation.low_confidence_mask, output_dir / "figures" / "orientation_low_confidence.png", title="Orientation low confidence", cmap="gray")

orientation.orientation_index.shape

## Export Summary

In [ ]:
summary_path = save_summary(
    output_dir,
    {
        "dataset": dataset.describe(),
        "virtual_images": {name: image.shape for name, image in virtual.images.items()},
        "fingerprints": {"shape": fingerprints.profiles.shape},
        "phase": {"labels_shape": phase.labels.shape, "candidate_scores": list(phase.candidate_scores)},
        "orientation": {"shape": orientation.orientation_index.shape, "roi": orientation.roi},
    },
)
summary_path